# GST Fraud Detection Pipeline
Step-by-step execution with column names printed after every step.

**Run this notebook from inside the `gst/` folder** (or adjust the `chdir` path below).

In [ ]:
import os, sys, pickle, shutil, uuid, time, warnings, concurrent.futures
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

warnings.filterwarnings('ignore')

# ── SET THIS to your absolute path of the gst/ folder ──────────────────────
GST_DIR = os.path.dirname(os.path.abspath('gst_fraud_pipeline_with_timer.py'))
os.chdir(GST_DIR)
sys.path.insert(0, GST_DIR)
sys.path.insert(0, os.path.dirname(GST_DIR))   # project root (for config/)

print(f'Working directory: {os.getcwd()}')

In [ ]:
# Import the pipeline class from the existing file
from gst_fraud_pipeline_with_timer import GSTAnalysis

analyzer = GSTAnalysis()
print(f'Output directory: {analyzer.output_dir}')

## Load Raw Data (before any step)

In [ ]:
raw_file = analyzer.find_input_file()
if raw_file.endswith('.parquet'):
    raw_df = pd.read_parquet(raw_file)
else:
    raw_df = pd.read_csv(raw_file)

print(f'Raw data shape: {raw_df.shape}')
print(f'\n=== RAW INPUT COLUMNS ({len(raw_df.columns)}) ===')
print(raw_df.columns.tolist())

## STEP 1 — Column Standardization
Saves → `final_output/gst_standardized.csv`

In [ ]:
success = analyzer.step1_column_standardization()
print(f'\nStep 1 success: {success}')

# ── PRINT COLUMNS AFTER STEP 1 ───────────────────────────────────────────────
df_step1 = pd.read_csv(analyzer._out('gst_standardized.csv'))
print(f'\n=== COLUMNS AFTER STEP 1 — Column Standardization ({len(df_step1.columns)}) ===')
print(df_step1.columns.tolist())

## STEP 2 — Data Validation & Cleaning
Saves → `final_output/gst_validated.csv`

In [ ]:
success = analyzer.step2_data_validation()
print(f'\nStep 2 success: {success}')

# ── PRINT COLUMNS AFTER STEP 2 ───────────────────────────────────────────────
df_step2 = pd.read_csv(analyzer._out('gst_validated.csv'))
print(f'\n=== COLUMNS AFTER STEP 2 — Data Validation ({len(df_step2.columns)}) ===')
print(df_step2.columns.tolist())

## STEP 3 — Rule Checking + Model Prediction (Parallel)
Saves → `final_output/gst_fraud_prediction.csv`

In [ ]:
success = analyzer.step3_parallel_rules_and_model()
print(f'\nStep 3 success: {success}')

# ── PRINT COLUMNS AFTER STEP 3 ───────────────────────────────────────────────
df_step3 = pd.read_csv(analyzer._out('gst_fraud_prediction.csv'))
print(f'\n=== COLUMNS AFTER STEP 3 — Rules + Model ({len(df_step3.columns)}) ===')
print(df_step3.columns.tolist())

## STEP 5 — Fraud Justification
Saves → MySQL table `gst_fraud_justification` (fallback: `gst_fraud_with_justification.csv`)

In [ ]:
success = analyzer.step5_fraud_justification()
print(f'\nStep 5 success: {success}')

# ── PRINT COLUMNS AFTER STEP 5 ───────────────────────────────────────────────
just_csv  = analyzer._out('gst_fraud_justification.csv')
just_parq = analyzer._out('gst_fraud_justification.parquet')
fallback  = analyzer._out('gst_fraud_with_justification.csv')

if os.path.exists(just_parq):
    df_step5 = pd.read_parquet(just_parq)
    print('Reading from: gst_fraud_justification.parquet')
elif os.path.exists(just_csv):
    df_step5 = pd.read_csv(just_csv)
    print('Reading from: gst_fraud_justification.csv')
elif os.path.exists(fallback):
    df_step5 = pd.read_csv(fallback)
    print('Reading from: gst_fraud_with_justification.csv (fallback)')
else:
    print('Justification file not found on disk — was saved to MySQL only.')
    df_step5 = None

if df_step5 is not None:
    print(f'\n=== COLUMNS AFTER STEP 5 — Fraud Justification ({len(df_step5.columns)}) ===')
    print(df_step5.columns.tolist())

## Column Summary Across All Steps

In [ ]:
print('=== COLUMN COUNT SUMMARY ===')
print(f'  Raw Input       : {len(raw_df.columns)} columns')
print(f'  After Step 1    : {len(df_step1.columns)} columns')
print(f'  After Step 2    : {len(df_step2.columns)} columns')
print(f'  After Step 3    : {len(df_step3.columns)} columns')
if df_step5 is not None:
    print(f'  After Step 5    : {len(df_step5.columns)} columns')

print('\n=== NEW COLUMNS ADDED AT STEP 3 (rule flags + model output) ===')
new_cols_step3 = [c for c in df_step3.columns if c not in df_step2.columns]
print(new_cols_step3)